# Clasificacion de Tumores Cerebrales - EfficientNet-B0

**Arquitectura:** EfficientNet-B0 (Red Neuronal Convolucional optimizada con Compound Scaling)

**Descripcion del experimento:** Entrenamiento y evaluacion del modelo EfficientNet-B0
para la clasificacion binaria de tumores cerebrales en imagenes de resonancia magnetica.
El experimento se ejecuta en dos fases: una fase base sin tecnicas de optimizacion
y una fase optimizada con data augmentation, early stopping, programador de learning rate
y precision mixta.

---
## 1. Montar Google Drive

Permite el acceso cuando Google lo solicite. Los archivos del proyecto
y el dataset se encuentran almacenados en Google Drive.

In [ ]:
# Montar Google Drive para acceder a los archivos del proyecto
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Configuracion del experimento

Definir el modelo a entrenar, la fase experimental y las rutas en Google Drive.

**Fase 1 - Base:** Sin data augmentation, sin early stopping, sin programador de LR,
sin precision mixta. Entrenamiento de 30 epocas fijas. Util para comparacion justa
entre arquitecturas.

**Fase 2 - Optimizado:** Con data augmentation, early stopping (paciencia de 7 epocas),
programador ReduceLROnPlateau, precision mixta (AMP) y weight decay. El entrenamiento
se detiene automaticamente cuando el modelo converge segun los criterios definidos
en el archivo de configuracion.

In [ ]:
# ===================================================
# CONFIGURACION DEL EXPERIMENTO
# ===================================================
# Nombre del modelo a entrenar
# Opciones: resnet50, efficientnet_b0, vit, swin, coatnet
MODELO = "efficientnet_b0"

# Fase experimental: 1 = Base, 2 = Optimizado
# (Ignorado si SCRATCH=True o NORMALIZED=True)
FASE = 2

# Entrenar desde cero sin pesos preentrenados
SCRATCH = True

# Pipeline de normalizacion estricta (elimina artefactos del dataset)
# CenterCrop cuadrado + RGB forzado + Equalize histograma
# Activar para TODOS los modelos para comparacion justa
NORMALIZED = True

# Ruta del proyecto en Google Drive
RUTA_DRIVE = "/content/drive/MyDrive/BrainTumor"

# Ruta del dataset en Google Drive
RUTA_DATASET = "/content/drive/MyDrive/BrainTumor_Dataset"
# ===================================================

# Mostrar la configuracion seleccionada
scratch_texto = "Si (desde cero)" if SCRATCH else "No (transfer learning)"
norm_texto = "Si (anti-artefactos)" if NORMALIZED else "No"
if NORMALIZED and SCRATCH:
    fase_texto = "Normalizado + Scratch (sin pesos, sin artefactos)"
elif NORMALIZED:
    fase_texto = "Normalizado (anti-artefactos)"
elif SCRATCH:
    fase_texto = "Scratch - sin pesos preentrenados"
else:
    fase_texto = "2 - Optimizado" if FASE == 2 else "1 - Base"
print(f"Modelo: {MODELO}")
print(f"Fase:   {fase_texto}")
print(f"Scratch: {scratch_texto}")
print(f"Normalizado: {norm_texto}")
print(f"Drive:  {RUTA_DRIVE}")
print(f"Dataset: {RUTA_DATASET}")


---
## 3. Copiar proyecto y dataset a Colab

Se copian los archivos desde Google Drive al entorno de Colab para acelerar
el acceso durante el entrenamiento. La carpeta de resultados se conserva
en ambos lugares.

In [ ]:
import os
import shutil

# Eliminar carpeta previa del proyecto si existe para evitar conflictos
if os.path.exists("/content/BrainTumor"):
    shutil.rmtree("/content/BrainTumor")

# Copiar el proyecto completo desde Google Drive
shutil.copytree(RUTA_DRIVE, "/content/BrainTumor")

# Cambiar al directorio del proyecto
os.chdir("/content/BrainTumor")
print("Proyecto copiado exitosamente")

# Copiar el dataset dentro de la estructura del proyecto
dst = "/content/BrainTumor/dataset/BrainTumor_Dataset"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(RUTA_DATASET, dst)
print("Dataset copiado exitosamente")

# Verificar que la estructura de datos sea la esperada
print(f"\nContenido de dataset/: {os.listdir('dataset')}")

---
## 4. Instalar dependencias

Google Colab incluye PyTorch y torchvision preinstalados.
Se instalan las librerias adicionales necesarias para el proyecto.

In [ ]:
# Instalar dependencias base del proyecto
!pip install -q matplotlib scikit-learn tqdm

# EfficientNet-B0 esta incluido en torchvision, no requiere dependencias adicionales
print("Dependencias instaladas correctamente")

---
## 5. Verificar GPU

Confirmar que el entorno de ejecucion de Colab tenga una GPU disponible.
Si no aparece ninguna GPU, ir a Entorno de ejecucion -> Cambiar tipo de entorno
de ejecucion y seleccionar T4 GPU.

In [ ]:
import torch

# Verificar version de PyTorch
print(f"PyTorch: {torch.__version__}")

# Verificar disponibilidad de CUDA
print(f"CUDA disponible: {torch.cuda.is_available()}")

# Mostrar nombre de la GPU si esta disponible
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No hay GPU disponible. El entrenamiento sera significativamente mas lento.")
    print("Se recomienda seleccionar T4 GPU en el menu Entorno de ejecucion.")

---
## 6. Analisis Exploratorio de Datos (EDA)

Antes de entrenar, se analiza la distribucion del dataset y se visualizan
ejemplos de imagenes de cada clase. Este paso permite identificar
desequilibrios entre clases o posibles anomalias en los datos.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Ruta raiz del dataset
dataset_root = Path("dataset/BrainTumor_Dataset")

# Contar la cantidad de imagenes por split y por clase
# Los splits (train, val, test) ya vienen predefinidos
print("=" * 50)
print("DISTRIBUCION DEL DATASET")
print("=" * 50)
total = 0
print(f"{'Split':<8} {'Clase':<6} {'Cant':<6} {'%':<8} {'Acum'}")
print("-" * 40)
for split in ["train", "val", "test"]:
    split_path = dataset_root / split
    classes = sorted(split_path.iterdir())
    counts = [len(list(c.iterdir())) for c in classes]
    split_total = sum(counts)
    for c, cnt in zip(classes, counts):
        pct = cnt / split_total * 100
        print(f"  {split:8s} / {c.name:6s}: {cnt:5d}  ({pct:5.1f}%)")
    total += split_total
print(f"  {'TOTAL':8s}: {total:5d} imagenes\n")
# Determinar el desbalance de clases por split
for split in ["train", "val", "test"]:
    split_path = dataset_root / split
    class_counts = {}
    for class_dir in sorted(split_path.iterdir()):
        class_counts[class_dir.name] = len(list(class_dir.iterdir()))
    max_cls = max(class_counts, key=class_counts.get)
    min_cls = min(class_counts, key=class_counts.get)
    max_c = max(class_counts.values())
    min_c = min(class_counts.values())
    ratio = class_counts[max_cls] / class_counts[min_cls]
    imbalance_pct = (max_c - min_c) / max_c * 100
    diff = max_c - min_c
    print(f"  {split:8s}: ratio {max_cls}/{min_cls} = {ratio:.2f}x  (desbalance = {imbalance_pct:.1f}%, {diff} img)")
print()

# Visualizar un ejemplo de cada clase (con tumor y sin tumor)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, clase in enumerate(["no", "yes"]):
    class_path = dataset_root / "train" / clase
    img_path = list(class_path.iterdir())[0]
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"Clase: {clase}\nDimensiones: {img.size[0]}x{img.size[1]} px")
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("results/eda_samples.png", bbox_inches="tight")
plt.show()
print("Imagenes de ejemplo guardadas en results/eda_samples.png")

---
## 7. Limpieza del dataset

Se verifica la integridad de todas las imagenes del dataset.
Se detectan imagenes corruptas, archivos con extensiones no validas
y se muestra la distribucion completa de clases por cada split.

In [ ]:
# Ejecutar el script de limpieza del dataset
!python src/clean_dataset.py

In [ ]:
# Verificar distribucion despues de la limpieza
from pathlib import Path
dataset_root = Path("dataset/BrainTumor_Dataset")
print("Distribucion final del dataset tras limpieza:")
print("-" * 55)
total_final = 0
for split in ["train", "val", "test"]:
    split_path = dataset_root / split
    classes = sorted(split_path.iterdir())
    counts = [len(list(c.iterdir())) for c in classes]
    split_total = sum(counts)
    for c, cnt in zip(classes, counts):
        print(f"  {split:8s} / {c.name:6s}: {cnt:5d}")
    print(f"  {'':8s}  {'Subtotal':6s}: {split_total:5d}")
    total_final += split_total
print(f"  {'TOTAL':8s}: {total_final:5d} imagenes")


---
## 8. Prueba rapida (2 epocas)

Antes del entrenamiento completo, se ejecuta una prueba rapida de 2 epocas
para verificar que todo el pipeline funciona correctamente:
- El modelo se carga correctamente
- Los dataloaders funcionan
- No hay errores de memoria
- El script de entrenamiento se ejecuta sin problemas

Tiempo estimado: 2 minutos.

In [ ]:
from pathlib import Path
if NORMALIZED and SCRATCH:
    phase_tag = "normalized_scratch"
elif NORMALIZED:
    phase_tag = "normalized"
elif SCRATCH:
    phase_tag = "scratch"
else:
    phase_tag = "optimized" if FASE == 2 else "base"
modelo_path = Path(f"results/{MODELO}/{phase_tag}/best_model.pth")
if modelo_path.exists():
    print(f"Saltando prueba rapida - ya existe {modelo_path}")
else:
    print("Ejecutando prueba rapida de 2 epocas...")
    scratch_flag = "--scratch" if SCRATCH else ""
    norm_flag = "--normalized" if NORMALIZED else ""
    !python src/train.py --model {MODELO} --phase {FASE} --epochs 2 {scratch_flag} {norm_flag}
    print("[OK] Prueba rapida completada" if modelo_path.exists() else "[ERROR] El entrenamiento fallo")


---
## 9. Entrenamiento completo

Ejecutar solo si la prueba rapida anterior finalizo correctamente.

**Fase 1 - Base:** Entrena durante 30 epocas fijas. Sin augmentation,
sin early stopping, sin programador de LR. Aproximadamente 30 minutos.

**Fase 2 - Optimizado:** Entrena hasta que el modelo converge. Los criterios
de convergencia incluyen: precision objetivo (99.5%), estabilizacion del loss,
early stopping con paciencia de 7 epocas y learning rate minimo.
Aproximadamente 30-60 minutos dependiendo del modelo.

In [ ]:
print(f"Iniciando entrenamiento completo de {MODELO}...")
scratch_flag = "--scratch" if SCRATCH else ""
norm_flag = "--normalized" if NORMALIZED else ""
!python src/train.py --model {MODELO} --phase {FASE} {scratch_flag} {norm_flag} && echo "[OK] Entrenamiento completado" || echo "[ERROR] El entrenamiento fallo"


---
## 10. Evaluacion del modelo

Se evalua el mejor modelo guardado durante el entrenamiento usando el
conjunto de prueba (test). Las metricas calculadas incluyen:

- **Accuracy:** Proporcion de predicciones correctas
- **Precision:** Proporcion de verdaderos positivos entre los positivos predichos
- **Recall (Sensitivity):** Proporcion de verdaderos positivos identificados
- **Specificity:** Proporcion de verdaderos negativos identificados
- **F1-Score:** Media armonica entre precision y recall
- **ROC-AUC:** Area bajo la curva ROC
- **Balanced Accuracy:** Promedio entre sensitivity y specificity
- **MCC:** Coeficiente de correlacion de Matthews
- **Parametros totales:** Cantidad de pesos del modelo
- **Tiempo de inferencia:** Tiempo promedio por batch

In [ ]:
print(f"Evaluando modelo {MODELO}...")
scratch_flag = "--scratch" if SCRATCH else ""
norm_flag = "--normalized" if NORMALIZED else ""
!python src/evaluate.py --model {MODELO} --phase {FASE} {scratch_flag} {norm_flag} && echo "[OK] Evaluacion completada" || echo "ERROR: La evaluacion fallo"


---
## 11. Visualizacion de resultados

Se muestran las metricas obtenidas y las graficas generadas durante la evaluacion.
Incluye la matriz de confusion y la curva ROC del modelo entrenado.

In [ ]:
import json

if NORMALIZED and SCRATCH:
    phase_tag = "normalized_scratch"
elif NORMALIZED:
    phase_tag = "normalized"
elif SCRATCH:
    phase_tag = "scratch"
else:
    phase_tag = "optimized" if FASE == 2 else "base"
ruta = Path(f"results/{MODELO}/{phase_tag}")

if (ruta / "metrics.json").exists():
    with open(ruta / "metrics.json") as f:
        metrics = json.load(f)
    print(f"Metricas de {MODELO} ({phase_tag}):")
    print("-" * 40)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:30s}: {v:.4f}")
        else:
            print(f"  {k:30s}: {v}")
else:
    print("No se encontraron metricas.")

from IPython.display import Image, display
for img in ["confusion_matrix.png", "roc_curve.png", "learning_curves.png"]:
    path = ruta / img
    if path.exists():
        print(f"\nMostrando {img}...")
        display(Image(filename=str(path)))


---
## 12. Guardar resultados en Google Drive

Los resultados del entrenamiento se copian de vuelta a Google Drive
para su preservacion una vez que la sesion de Colab se cierre.
Esto incluye los pesos del modelo, las metricas, el historial
de entrenamiento y las graficas generadas.

In [ ]:
if NORMALIZED and SCRATCH:
    phase_tag = "normalized_scratch"
elif NORMALIZED:
    phase_tag = "normalized"
elif SCRATCH:
    phase_tag = "scratch"
else:
    phase_tag = "optimized" if FASE == 2 else "base"
src = f"/content/BrainTumor/results/{MODELO}/{phase_tag}"
dst = f"{RUTA_DRIVE}/results/{MODELO}/{phase_tag}"
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Resultados guardados en: {dst}")
print("La sesion de Colab ya puede cerrarse de forma segura.")
